# Validate and inspect Bundestag 2021 constituency vote data

This notebook reruns the same Python pipeline without writing files. It is intended for inspecting exact margins, generated row counts and data volume before the JSON files are moved into the application.

In [ ]:
from pathlib import Path
import sys
import pandas as pd


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository checkout")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))

In [ ]:
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw21_wbz_ergebnisse.csv"
STATE_DEMOGRAPHICS_CSV = ROOT / "scripts/data/btw21_rws_stimmabgabe_laender.csv"
FEDERAL_METHOD_DEMOGRAPHICS_CSV = None

In [ ]:
from scripts.election_data import prepare_btw2021_vote_entries

result = prepare_btw2021_vote_entries(
    district_results_csv=DISTRICT_RESULTS_CSV,
    state_demographics_csv=STATE_DEMOGRAPHICS_CSV,
    federal_method_demographics_csv=FEDERAL_METHOD_DEMOGRAPHICS_CSV,
)
result.validation

## Source coverage

In [ ]:
pd.DataFrame(
    {
        "value": [
            result.districtTotals["districtId"].nunique(),
            result.districtTotals["state"].nunique(),
            result.districtTotals["party"].nunique(),
            len(result.firstVotes),
            len(result.secondVotes),
        ]
    },
    index=["constituencies", "states", "parties", "first-vote rows", "second-vote rows"],
)

## Profile fallbacks

In [ ]:
result.profiles.groupby(
    ["demographicProfileSource", "methodSeedSource"]
)["party"].nunique().sort_values(ascending=False)

## Estimated JSON size

This serializes in memory only, so it can be used to judge whether the complete party coverage is acceptable for the browser before writing the files.

In [ ]:
import json
from dataclasses import asdict

first_bytes = len(json.dumps([asdict(entry) for entry in result.firstVotes], ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
second_bytes = len(json.dumps([asdict(entry) for entry in result.secondVotes], ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
{
    "first_votes_mib": first_bytes / 1024**2,
    "second_votes_mib": second_bytes / 1024**2,
    "combined_mib": (first_bytes + second_bytes) / 1024**2,
}

## Example constituency

In [ ]:
sample_district = int(result.districtTotals["districtId"].min())
[
    entry
    for entry in result.firstVotes
    if entry.districtId == sample_district
][:24]